# 📓 Notebook 01 — Validação de Dados Pré-Treinamento

> **Pergunta operacional:** o dado que chegou está pronto para virar treino?
>
> **Origem canônica:** [Disciplina 01, Aula 01 — Entendimento do Problema de Negócio e Dados](../../../../disciplinas/01-introducao-ao-ciclo-de-vida-de-modelos/temas.md) (temas 1.2–1.4).

## Objetivo

Construir um **contrato de dados explícito** para o dataset Telco Customer Churn e usá-lo como porta de entrada do pipeline. Saímos deste notebook com:

1. esquema declarativo (tipos, ranges, nulidade, valores permitidos);
2. invariantes de prontidão (balanceamento, duplicatas, leakage óbvio);
3. relatório de readiness assinável por engenharia e negócio.

## Por que isso importa

Modelos de churn fracassam mais por **dados ruins** do que por arquitetura mal escolhida. O contrato funciona como uma cláusula "não rodaremos treino se isto aqui não passar". Sem ele, o erro reaparece como métrica suspeita semanas depois — e ninguém mais sabe de onde veio.

## 0. Setup

Esta célula garante o ambiente, importa as bibliotecas e carrega o dataset. Se `pandera` não estiver instalado, caímos no caminho pandas puro — o conteúdo conceitual segue idêntico.

In [1]:
from __future__ import annotations

import sys
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import pandera as pa
    from pandera import Column, Check
    PANDERA_AVAILABLE = True
except ImportError:
    PANDERA_AVAILABLE = False

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = Path("../dataset/processed/telco_churn.csv")
if not DATA_PATH.exists():
    print("⚠️  Dataset não encontrado. Rode: python ../dataset/generate_dataset.py")
    sys.exit(1)

df = pd.read_csv(DATA_PATH)
print(f"Linhas: {len(df):,} | Colunas: {df.shape[1]}")
df.head(3)

Linhas: 8,000 | Colunas: 21


,customer_id,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn
0,C0000000,Male,0,No,No,37,Yes,Yes,DSL,No,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.14,2229.28,No
1,C0000001,Female,1,Yes,No,11,Yes,No,Fiber optic,No,...,Yes,Yes,No,No,One year,No,Credit card (automatic),46.56,517.43,No
2,C0000002,Male,0,Yes,Yes,22,Yes,No,DSL,No,...,Yes,No,Yes,No,Month-to-month,No,Bank transfer (automatic),66.47,1460.29,Yes


## 1. Definindo o contrato de dados

O contrato responde três perguntas:

| Pergunta | O que checar |
|----------|--------------|
| O dado **chegou bem-formado?** | tipos, presença de colunas, valores categóricos permitidos |
| O dado **faz sentido de negócio?** | ranges plausíveis, regras inter-coluna |
| O dado **dá para treinar?** | nulidade, duplicidade, balanceamento, leakage |

Vamos modelar essas 3 camadas com `pandera` (caminho preferido) e replicar manualmente quando indisponível.

In [2]:
BINARY_YES_NO = ["Yes", "No"]
INTERNET_GATED = ["Yes", "No", "No internet service"]

if PANDERA_AVAILABLE:
    schema = pa.DataFrameSchema(
        {
            "customer_id": Column(str, Check.str_startswith("C")),
            "gender": Column(str, Check.isin(["Female", "Male"])),
            "senior_citizen": Column(int, Check.isin([0, 1])),
            "partner": Column(str, Check.isin(BINARY_YES_NO)),
            "dependents": Column(str, Check.isin(BINARY_YES_NO)),
            "tenure": Column(int, Check.in_range(0, 72)),
            "phone_service": Column(str, Check.isin(BINARY_YES_NO)),
            "multiple_lines": Column(str, Check.isin(["Yes", "No", "No phone service"])),
            "internet_service": Column(str, Check.isin(["DSL", "Fiber optic", "No"])),
            "online_security": Column(str, Check.isin(INTERNET_GATED)),
            "online_backup": Column(str, Check.isin(INTERNET_GATED)),
            "device_protection": Column(str, Check.isin(INTERNET_GATED)),
            "tech_support": Column(str, Check.isin(INTERNET_GATED)),
            "streaming_tv": Column(str, Check.isin(INTERNET_GATED)),
            "streaming_movies": Column(str, Check.isin(INTERNET_GATED)),
            "contract": Column(str, Check.isin(["Month-to-month", "One year", "Two year"])),
            "paperless_billing": Column(str, Check.isin(BINARY_YES_NO)),
            "payment_method": Column(str),
            "monthly_charges": Column(float, Check.in_range(0.0, 200.0)),
            "total_charges": Column(float, Check.greater_than_or_equal_to(0.0)),
            "churn": Column(str, Check.isin(BINARY_YES_NO)),
        },
        strict=True,
        coerce=True,
    )
    validated = schema.validate(df, lazy=True)
    print("✅ pandera: esquema validado com sucesso.")
else:
    print("ℹ️  pandera não instalado — caindo para validação manual abaixo.")

✅ pandera: esquema validado com sucesso.


C:\Users\ricar\Github\mlet\fases\fase-01-produtizacao-de-modelos\eventos\grupos-de-estudo\encontro-01\.venv\Lib\site-packages\pandera\_pandas_deprecated.py:143: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


### 🛑 Breakpoint — discussão

1. Quais colunas têm **regras inter-coluna** (uma depende da outra)?
2. O contrato proíbe valores nulos. Isso é certo para um dataset de churn?
3. Onde colocar o contrato no pipeline: na ingestão, no treino, ou em ambos?

## 2. Invariantes de prontidão (data readiness)

Schema só checa forma. Prontidão checa **se o dado serve para o problema**. A função abaixo é o coração do data readiness checklist da Aula 01 (volume, qualidade, distribuição, governança).

In [3]:
@dataclass(slots=True)
class ReadinessReport:
    passed: bool
    rows: int
    duplicates: int
    missing_per_column: dict[str, int]
    class_balance: dict[str, float]
    issues: list[str] = field(default_factory=list)


def evaluate_readiness(df: pd.DataFrame, target: str = "churn", min_rows: int = 5_000) -> ReadinessReport:
    issues: list[str] = []

    if len(df) < min_rows:
        issues.append(f"volume baixo: {len(df)} < {min_rows}")

    duplicates = int(df.duplicated(subset="customer_id").sum())
    if duplicates:
        issues.append(f"{duplicates} customer_id duplicados")

    missing = df.isna().sum().to_dict()
    if any(v > 0 for v in missing.values()):
        nz = {k: int(v) for k, v in missing.items() if v > 0}
        issues.append(f"colunas com missing: {nz}")

    balance = df[target].value_counts(normalize=True).to_dict()
    minority = min(balance.values())
    if minority < 0.10:
        issues.append(f"classe minoritária muito rara ({minority:.1%})")

    return ReadinessReport(
        passed=not issues,
        rows=len(df),
        duplicates=duplicates,
        missing_per_column={k: int(v) for k, v in missing.items()},
        class_balance={k: float(v) for k, v in balance.items()},
        issues=issues,
    )

report = evaluate_readiness(df)
print("Pronto para treino:", "✅" if report.passed else "❌")
print("Distribuição da classe:", {k: round(v, 3) for k, v in report.class_balance.items()})
if report.issues:
    for issue in report.issues:
        print(" -", issue)

Pronto para treino: ✅
Distribuição da classe: {'No': 0.735, 'Yes': 0.265}


## 3. Detecção de leakage óbvio

Leakage é a falha mais cara de descobrir tarde. A heurística mais simples: **se uma feature tem correlação altíssima com o alvo, ou se foi medida depois do evento de churn, ela vaza**. Aqui usamos um teste rápido de associação categórica + correlação numérica para sinalizar candidatos suspeitos.

In [4]:
from sklearn.preprocessing import LabelEncoder

y = (df["churn"] == "Yes").astype(int)

suspects: list[tuple[str, float]] = []
for col in df.columns.drop(["churn", "customer_id"]):
    series = df[col]
    if series.dtype == object:
        codes = LabelEncoder().fit_transform(series.astype(str))
    else:
        codes = series.to_numpy()
    corr = float(np.corrcoef(codes, y)[0, 1])
    if abs(corr) > 0.55:
        suspects.append((col, corr))

if suspects:
    print("⚠️  Possíveis leakages (|corr| > 0.55):")
    for col, c in suspects:
        print(f"  - {col:<22} corr={c:+.3f}")
else:
    print("✅ Nenhuma feature com correlação suspeita com o alvo.")

✅ Nenhuma feature com correlação suspeita com o alvo.


## 4. Relatório final de readiness

Em produção, este relatório seria um JSON publicado pelo job de ingestão e consumido pelo pipeline de treino. Aqui imprimimos no notebook para discussão.

In [5]:
import json

report_payload = {
    "dataset": "telco_churn",
    "rows": report.rows,
    "duplicates": report.duplicates,
    "class_balance": report.class_balance,
    "issues": report.issues,
    "schema_validated": PANDERA_AVAILABLE,
    "verdict": "GO" if report.passed else "NO-GO",
}
print(json.dumps(report_payload, indent=2, ensure_ascii=False))

{
  "dataset": "telco_churn",
  "rows": 8000,
  "duplicates": 0,
  "class_balance": {
    "No": 0.735,
    "Yes": 0.265
  },
  "issues": [],
  "schema_validated": true,
  "verdict": "GO"
}


## 🧠 Exercícios

**Iniciante.** Inclua no contrato a regra: `total_charges == 0` ⇒ `tenure == 0`. Use `pa.Check` ou pandas e mostre quais registros violam.

**Intermediário.** Substitua a heurística de leakage por **Mutual Information** (`sklearn.feature_selection.mutual_info_classif`). Quais features sobem ou somem da lista?

**Avançado.** Refaça `evaluate_readiness` para gerar um *Markdown Card* assinável (autor, dataset version hash, verdict). Persista em `dataset/processed/readiness_card.md` e cite a versão exata do dataset no Model Card do Tech Challenge.

## ➡️ Para o próximo notebook

Agora que o dado **passou no portão**, vamos discutir **onde** corrigir o que ainda não está perfeito: na camada de dados (medalhão) ou dentro do pipeline de treino. Abra [`02_golden_layer_vs_sklearn_pipelines.ipynb`](02_golden_layer_vs_sklearn_pipelines.ipynb).